# `%provenance` magic demo

Ask for citations from inside the notebook you are working in, instead of calling the workflow functions directly.

Run this in the `lang` conda env. The cells below are not pre-run: `%provenance` calls Gemini to route the request, so the output depends on your `GOOGLE_API_KEY` in `src/.env`.

## Load the extension

`provenance` is a top-level module of the installed package (`pip install -e ".[dev]"`), so `%load_ext` finds it with nothing on `sys.path`.

In [ ]:
%load_ext provenance

## Point it at a notebook

`%provenance` auto-detects the running notebook via `ipynbname`. That works in classic Jupyter but usually fails in VSCode, because it matches the kernel against the Jupyter server's session list. Set the target explicitly with `%provenance_notebook`. Both requests inject cells **in place**, so we point it at a throwaway copy of the example:

In [ ]:
import shutil
shutil.copy('../examples/paleoPCAlite.ipynb', '../examples/paleoPCAlite_demo.ipynb')
%provenance_notebook ../examples/paleoPCAlite_demo.ipynb

## Cite the software

Routes to `cite_software`, which extracts imports with AST and injects a single cell that builds a pandas DataFrame of the citation metadata (from the local `Citations/` files, parsed with bibtexparser). It reports the libraries covered; reload the target and run the injected cell to see the DataFrame.

In [ ]:
%provenance cite the software

### One specific library

The agent passes the library name through as a filter.

In [ ]:
%provenance cite Pyleoclim

### Format

There is no APA-vs-BibTeX choice on the software path any more - `cite_software` always injects a metadata DataFrame - so naming a format in the request is ignored here. (Format still applies to the data workflow.)

In [ ]:
%provenance cite the software in BibTeX

## Cite the datasets

Routes to `cite_data`, which detects dataset variables with the LLM and **injects a retrieval cell per dataset into the notebook file**.

The citations are the output of those injected cells, not of this one - retrieval needs the live objects already loaded in that notebook's kernel. So the magic reports what it wrote, and you reload the target notebook and run the new cells.

Note it writes **in place**. Reload the target notebook before editing it further, or an editor save can overwrite the injected cells.

In [ ]:
%provenance cite the datasets

## Notes

- Every workflow reads the `.ipynb` **from disk**. A cell you typed but have not saved is invisible, so save before asking.
- An unroutable request (`%provenance what is the weather`) prints what the two tools cover rather than failing silently.
- Software citations are fully offline: `cite_software` reads the local `Citations/` files (parsed with bibtexparser) and injects the metadata cell without calling Gemini. APA rendering via Gemini now applies only to the data workflow's cell; issue #29 tracks replacing it with a deterministic formatter.